In [11]:
"""
Question 1: Data Cleaning and Basic Analysis

Using the provided dataset 'sales_data.csv', perform the following tasks:

1. Load and examine the dataset
2. Clean the data:
   - Handle missing values in the 'price' column (replace with mean)
   - Handle missing values in the 'category' column (replace with mode)
   - Convert 'sale_date' to datetime
3. Create a summary of:
   - Total number of sales per category
   - Average price per category
   - Number of missing values handled

The cleaned dataset should be ready for further analysis.
"""

"\nQuestion 1: Data Cleaning and Basic Analysis\n\nUsing the provided dataset 'sales_data.csv', perform the following tasks:\n\n1. Load and examine the dataset\n2. Clean the data:\n   - Handle missing values in the 'price' column (replace with mean)\n   - Handle missing values in the 'category' column (replace with mode)\n   - Convert 'sale_date' to datetime\n3. Create a summary of:\n   - Total number of sales per category\n   - Average price per category\n   - Number of missing values handled\n\nThe cleaned dataset should be ready for further analysis.\n"

In [12]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats as stats

In [13]:
df = pd.read_csv("../data/sales_data.csv")
df

,sale_id,sale_date,category,price,quantity
0,1,2023-01-01,Home & Garden,NaN,6
1,2,2023-01-02,Sports,121.772999,7
2,3,2023-01-03,Books,114.430277,4
3,4,2023-01-04,Sports,106.716521,7
4,5,2023-01-05,Sports,76.285766,9
...,...,...,...,...,...
995,996,2025-09-22,Clothing,53.104276,3
996,997,2025-09-23,Electronics,126.493293,4
997,998,2025-09-24,Electronics,97.664884,1
998,999,2025-09-25,Home & Garden,94.585603,9


### Question 1

In [14]:
# First we visualize the data, to understand what the data says, and what information we expect to get from it
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   sale_id    1000 non-null   int64  
 1   sale_date  1000 non-null   object 
 2   category   970 non-null    object 
 3   price      951 non-null    float64
 4   quantity   1000 non-null   int64  
dtypes: float64(1), int64(2), object(2)
memory usage: 39.2+ KB


,sale_id,price,quantity
count,1000.000000,951.000000,1000.000000
mean,500.500000,101.883052,5.033000
std,288.819436,30.379179,2.590487
min,1.000000,12.359485,1.000000
25%,250.750000,81.635403,3.000000
50%,500.500000,101.314344,5.000000
75%,750.250000,121.246977,7.000000
max,1000.000000,195.793227,9.000000


In [15]:
# This will be a dummy dataset we use to test our modifications, to not mess with the actual dataset.
# This may become unviable for larger datasets, but this one is pretty tame.
t_df = df.copy()

In [16]:
# Compute the number of missing values that needs to be handled:
number_of_missing = {col: t_df[col].isna().sum() for col in t_df.columns if t_df[col].isna().sum()!=0}
number_of_missing

{'category': 30, 'price': 49}

In [17]:
# Replacing the missing values in the 'price' column with the mean price.
mean_price = df['price'].mean()
t_df['price'] = t_df['price'].fillna(mean_price)

# Replacing the missing values in the 'category' column with the mode category.
mode_cat = df['category'].mode()[0]
t_df['category'] = t_df['category'].fillna(mode_cat)

# Converting the sale date to "DateTime"
t_df['sale_date'] = pd.to_datetime(t_df['sale_date'])

# Check .info() again to me sure we have everything correct. Every column should now be 1000 non-null, and sale_date should be 'datetime'
t_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   sale_id    1000 non-null   int64         
 1   sale_date  1000 non-null   datetime64[ns]
 2   category   1000 non-null   object        
 3   price      1000 non-null   float64       
 4   quantity   1000 non-null   int64         
dtypes: datetime64[ns](1), float64(1), int64(2), object(1)
memory usage: 39.2+ KB


In [18]:
# Group by category to then have the total amount of each category sold
sales_by_cat = t_df.groupby('category')
sales_by_cat['quantity'].describe() # This gives us some good info, such as the mean quantity bought (per sale), minimum, maximum, etc

# If we just want the count, then we can do:
total_sales_by_cat = sales_by_cat['quantity'].describe()['count']
total_sales_by_cat

category
Books            179.0
Clothing         185.0
Electronics      238.0
Home & Garden    201.0
Sports           197.0
Name: count, dtype: float64

In [19]:
# To get the average price per category, we can use our 'sales_by_cat' aggregate dataframe:
sales_by_cat['price'].describe() # This, again, gives us more info than just the average price per category, which can be useful

# If we just want the average price per sale on each category:
avg_price_per_category = sales_by_cat['price'].describe()['mean']
avg_price_per_category

category
Books            101.695811
Clothing         103.707144
Electronics      100.533731
Home & Garden    102.614946
Sports           101.223595
Name: mean, dtype: float64

In [20]:
# Saving it to a new .csv:
path = "../data/clean_sales_data.csv"

t_df.to_csv(path)